# Quantum Relief - walkthrough
Choose 4 of 10 warehouse sites to stock, trading people reached against flood risk. Each step below maps to one module in `src/`.

In [1]:
import sys; sys.path.insert(0, '..')   # make the src package importable
import numpy as np
from src.instance import make_instance
from src.baseline import brute_force, greedy_top_k, describe
from src.qaoa import run_qaoa, sample_counts, build_circuit, dicke_circuit
from src.evaluate import problem, metrics, counts_to_probs
from src.formulation import index_to_bits, qubo_matrix, to_ising, default_scale

## 1. The instance
`mu` is the average number of people each site reaches (thousands). `sigma` records which sites fail together.

In [2]:
inst = make_instance()          # seeded: same numbers every run
for name, basin, m in zip(inst.names, inst.basins, inst.mu):
    print(f'{name:<13}{basin:<12}{m:6.1f}')

Nowshera     Kabul         70.3
Charsadda    Kabul         67.7
Sukkur       Lower Indus   72.6
Larkana      Lower Indus   64.1
Dadu         Lower Indus   61.2
Rajanpur     Mid Indus     58.4
D.G. Khan    Mid Indus     58.4
Muzaffargarh Chenab        56.5
Jhang        Chenab        53.8
Multan       Chenab        54.2


## 2. Exact and greedy references

In [3]:
LAM = 0.005                      # caution level (risk weight)
names = lambda x: ', '.join(n for n, v in zip(inst.names, x) if v)
greedy = greedy_top_k(inst.mu, inst.k)
best_cost, best = brute_force(inst.mu, inst.sigma, LAM, inst.k)[0]
for label, x in (('greedy', greedy), ('exact ', best)):
    mean, std = describe(x, inst.mu, inst.sigma)
    print(f'{label}: reach {mean:.1f}k, std {std:.1f}k -> {names(x)}')

greedy: reach 274.7k, std 108.6k -> Nowshera, Charsadda, Sukkur, Larkana
exact : reach 255.5k, std 72.3k -> Nowshera, Sukkur, D.G. Khan, Multan


## 3. The quantum circuit
Dicke state (only valid plans), then a cost layer and an XY mixer. Drawn here for a 4-site toy so it fits on screen.

In [4]:
q, c = qubo_matrix(inst.mu[:4], inst.sigma[:4, :4], LAM)
ham, _ = to_ising(q / default_scale(inst.mu[:4], inst.sigma[:4, :4], LAM), 0)
toy, _, _ = build_circuit(ham, k=2, p=1, method='xy')
print(toy.draw(fold=110))

                                                                                  ┌───────────┐          »
q_0: ──────────────────────────────────────────────────────────────────────────■──┤ Ry(1.231) ├──■────■──»
                                     ┌─────────┐          ┌────────────┐       │  └─────┬─────┘  │  ┌─┴─┐»
q_1: ─────────────────────────────■──┤ Ry(π/2) ├──■────■──┤ Ry(1.9106) ├──■────┼────────■────────┼──┤ X ├»
     ┌───┐     ┌──────────┐       │  └────┬────┘  │  ┌─┴─┐└─────┬──────┘┌─┴─┐┌─┴─┐      │      ┌─┴─┐└───┘»
q_2: ┤ X ├──■──┤ Ry(2π/3) ├──■────┼───────■───────┼──┤ X ├──────■───────┤ X ├┤ X ├──────■──────┤ X ├─────»
     ├───┤┌─┴─┐└────┬─────┘┌─┴─┐┌─┴─┐     │     ┌─┴─┐└───┘              └───┘└───┘             └───┘     »
q_3: ┤ X ├┤ X ├─────■──────┤ X ├┤ X ├─────■─────┤ X ├────────────────────────────────────────────────────»
     └───┘└───┘            └───┘└───┘           └───┘                                                    »
«     ┌─────────┐                    

## 4. Train and measure (full 10-qubit problem)

In [5]:
prob = problem(inst.mu, inst.sigma, LAM, inst.k)
for p in (1, 2, 3):
    res = run_qaoa(inst.mu, inst.sigma, LAM, inst.k, method='xy', p=p, objective='cvar')
    m = metrics(counts_to_probs(sample_counts(res), inst.n), prob)
    print(f"p={p}: valid shots {m['feasible_rate']:.0%}, ratio {m['approx_ratio']:.2f}, "
          f"P(optimum) {m['p_optimum']:.1%} ({m['amplification']:.0f}x random), pick rank #{m['pick_rank']}")

p=1: valid shots 100%, ratio 0.75, P(optimum) 2.6% (5x random), pick rank #1


p=2: valid shots 100%, ratio 0.80, P(optimum) 5.6% (12x random), pick rank #1


p=3: valid shots 100%, ratio 0.79, P(optimum) 9.5% (20x random), pick rank #1


## 5. The answer

In [6]:
pick = index_to_bits(m['pick'], inst.n)
print('quantum plan:', names(pick))
print('matches exact optimum:', bool((pick == best).all()))

quantum plan: Nowshera, Sukkur, D.G. Khan, Multan
matches exact optimum: True
